## Business Context
Olist is a Brazilian marketplace that connects small sellers with customers, which generates a huge amount of data through customers, orders, sellers, payments, deliveries and customer reviews. This information of about 100k orders from 2016- 2018 is stored across 9 related datasets, which makes it hard to understand overall business performance and customer behaviour from raw data alone. 

The objective is to process, analyze and generate meaningful insights on sales, customers, delivery and satisfaction to improve the business performance.

## Business Objectives

1. E-Commerce Performance Monitoring 
2. Customer Behavior & Segmentation 
3. Sales & Revenue Optimization 
4. Product & Seller Performance Analysis 
5. Delivery & Operational Optimization 
6. Customer Experience Improvement 
7. Data-Driven Business Decision Making 

## Analysis Questions

**Business Overview**

1. What are the overall KPIs: total revenue, total orders, total customers, total sellers, average order value and average review score?

**Sales Analysis**

2. How does revenue change month over month (year–month trend)?
3. Which product categories generate the most revenue?
4. Which products are the top sellers by quantity and by revenue?
5. How are sales distributed across states and cities?
6. Which payment methods are used most, and how does order value vary by payment method?

**Customer Analysis**

7. How are customers distributed across states?
8. How many customers are repeat buyers vs one-time buyers?
9. Who are the top customers by total spending and by number of items purchased?
10. Do customers who experienced a delivery delay place fewer repeat orders than those who did not?
11. How many customers purchased in more than one year?

**Seller & Product Analysis**

12. Who are the top sellers by revenue and by order count?
13. What is the average review rating for each seller?
14. How much revenue does each seller generate per product category?
15. Which categories perform best and worst in terms of revenue and ratings?

**Delivery Analysis**

16. What is the average delivery time?
17. How many orders are delivered on time vs late, overall and by seller?
18. How does delivery performance vary across states?
19. How does the length of the delivery delay relate to the review score?

**Customer Experience**

20. How are review scores distributed?
21. How do review scores vary across product categories and sellers?

**Statistical Questions**

22. Do delayed orders receive significantly lower review scores than on-time orders?
23. Does the average order value differ significantly across product categories?
24. Is there a significant association between payment method and order status?

*Additional questions may be explored during the analysis.**

## Data Dictionary

Provides description and data types for each column from all 9 datasets along with relationship between the tables.

[Open data dictionary](../docs/data_dictionary.xlsx)

In [17]:
# reading data dictionary file
import pandas as pd
data_dict=pd.read_excel(r"F:\Guvi\cart2insights\docs\data_dictionary.xlsx")
data_dict

,Table,Column,Raw dtype,Final dtype,Description,Key
0,customers,customer_id,object,object,customer id created for each order,primary Key
1,customers,customer_unique_id,object,object,"unique id of the actual customer, same across ...",none
2,customers,customer_zip_code_prefix,int64,object,first 5 digits of the customer's postal code,join Key (geolocation)
3,customers,customer_city,object,object,city the customer belongs to,none
4,customers,customer_state,object,object,state the customer belongs to,none
5,geolocation,geolocation_zip_code_prefix,int64,object,first 5 digits of the postal code (it repeats ...,none
6,geolocation,geolocation_lat,float64,float64,latitude of a point of that zip code area,none
7,geolocation,geolocation_lng,float64,float64,longitude of a point of that zip code area,none
8,geolocation,geolocation_city,object,object,"city that zip code, lat and long belongs to",none
9,geolocation,geolocation_state,object,object,"state that zip code, lat and long belongs to",none


## Loading raw data
This code downloads all 9 csv files present in the shared google drive folder if they are not existed already in ""../data/raw/csv_files"" this path.
This will download the files only in the first run, on later runs only files are loaded from the local folder.

In [ ]:
# to download google files and folders, install gdown
%pip install gdown

In [22]:
import os
import gdown
import pandas as pd

# storing google drive folder id
folder_id= "15T53qM0tcIsSs3REAcrDqdHzQ5wif6lc"

# relative path to download the raw csv files
data_path= "../data/raw/csv_files"

# checking whether the path exists already, if not it'll download all the files from the folder_id
if not os.path.exists(data_path):
    gdown.download_folder(id=folder_id, output=data_path, quiet= False)

# reading all the csv files
customers= pd.read_csv(data_path + "/olist_customers_dataset.csv")
geolocation= pd.read_csv(data_path + "/olist_geolocation_dataset.csv")
order_items= pd.read_csv(data_path + "/olist_order_items_dataset.csv")
order_payments= pd.read_csv(data_path + "/olist_order_payments_dataset.csv")
order_reviews= pd.read_csv(data_path + "/olist_order_reviews_dataset.csv")
orders= pd.read_csv(data_path + "/olist_orders_dataset.csv")
products= pd.read_csv(data_path + "/olist_products_dataset.csv")
sellers= pd.read_csv(data_path + "/olist_sellers_dataset.csv")
category_translation= pd.read_csv(data_path + "/product_category_name_translation.csv")

In [18]:
# Understanding the structue of each table
customers.head()
#geolocation.head()
#order_items.head()
#order_payments.head()
#order_reviews.head()
#orders.head()
#products.head()
#sellers.head()
#category_translation.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [19]:
# making a dictionary of all the files to do certain operations combinely
dfs={"customers":customers,
     "geolocation":geolocation,
     "order_items": order_items,
     "order_payments": order_payments,
     "order_reviews": order_reviews,
     "orders": orders,
     "products": products,
     "sellers": sellers,
     "category_translation": category_translation}

In [20]:
# checking the shape of all the files
# extracted shapes for all the files and then converted to DataFrame and transposed for the better visibility
shapes= pd.DataFrame({name:df.shape for name,df in dfs.items()},index=['rows','columns']).T
shapes

,rows,columns
customers,99441,5
geolocation,1000163,5
order_items,112650,7
order_payments,103886,5
order_reviews,99224,7
orders,99441,8
products,32951,9
sellers,3095,4
category_translation,71,2


In [21]:
# checking the dtypes of columns in all the files
# df.dtypes extracts dtypes for all the files with 9 different series and concat is used to stack all the series together
dtypes= pd.concat({name:df.dtypes for name,df in dfs.items()})
dtypes

customers             customer_id                       object
                      customer_unique_id                object
                      customer_zip_code_prefix           int64
                      customer_city                     object
                      customer_state                    object
geolocation           geolocation_zip_code_prefix        int64
                      geolocation_lat                  float64
                      geolocation_lng                  float64
                      geolocation_city                  object
                      geolocation_state                 object
order_items           order_id                          object
                      order_item_id                      int64
                      product_id                        object
                      seller_id                         object
                      shipping_limit_date               object
                      price                            